# Big Data Analysis — Group Project
## Notebook 3: Deep Learning for Sentiment (DistilBERT)

In Notebook 2 we built a Logistic Regression classifier on TF-IDF features that
hit AUC ~0.94 on held-out reviews. Pretty solid for such a simple model.

So why bother with deep learning? Because bag-of-words approaches like TF-IDF
have known weaknesses:
- **Negation**: "not bad" reads as negative to BoW (it sees "bad"), but it actually
  means mildly positive.
- **Context**: "this product is the bomb" → BoW sees "bomb" and might lean negative.
- **Word order and dependencies**: BoW throws away order entirely.

A transformer like BERT reads the whole sentence bidirectionally and learns
contextual representations of each word. For sentiment, that often matters.

### What this notebook does
1. Loads our cleaned data and the LR pipeline saved by Notebook 2.
2. Uses **DistilBERT** (a smaller, faster BERT variant) pretrained on SST-2 movie
   reviews, applied at scale via Spark's `predict_batch_udf`.
3. Compares DistilBERT vs LR on the same test sample.
4. Looks at the disagreements — where does each model win.
5. Aggregates sentiment per product (the Lab 9 "per play" pattern adapted to our data).
6. Mini-experiment: uses BERT confidence as a feature for helpfulness prediction.

### Important caveat about scaling
DistilBERT inference on CPU takes a few ms per review. Running it on all ~500k
reviews would take hours. We sample down to ~5,000 reviews for the comparison.
On a real cluster with GPUs we'd run the full dataset. We flag this explicitly
where it matters — the brief specifically asks for this kind of awareness.

---
## 1. Setup

Same Spark + Java install as before, plus the deep learning dependencies from
Lab 9: `torch`, `transformers`, `pyarrow`.

In [1]:
!pip install pyspark torch transformers pyarrow --quiet

In [2]:
# Java 17 — skip if already installed
!sudo apt-get update -qq
!sudo apt-get install -y openjdk-17-jdk-headless -qq

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator, MulticlassClassificationEvaluator
)

# Note: bumping driver memory because the transformer model + tokeniser pin
# significant memory on the worker side
spark = (
    SparkSession.builder
        .master("local[*]")
        .appName("BigDataProject_DL")
        .config("spark.driver.memory", "8g")
        .config("spark.executor.memory", "4g")
        .config("spark.sql.shuffle.partitions", "16")
        .config("spark.sql.adaptive.enabled", "true")
        .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} ready.")

Spark 4.0.2 ready.


In [4]:
# Mount Drive — same project folder as Notebooks 1 and 2
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---
## 2. Loading the Data and Notebook 2's Pipeline

We load:
- The cleaned reviews from Notebook 1.
- The trained sentiment pipeline from Notebook 2 (TF-IDF + LR).

Being able to reload a saved `PipelineModel` is one of the main points of using
Pipelines — the preprocessing logic travels with the model, so there's no risk of
training-vs-serving skew.

In [5]:
CLEANED_PATH = f"{PROJECT_DIR}/reviews_cleaned.parquet"
MODEL_PATH   = f"{PROJECT_DIR}/sentiment_pipeline_model"

reviews_df = spark.read.parquet(CLEANED_PATH)
print(f"Total reviews: {reviews_df.count():,}")

# Reload the pipeline trained in Notebook 2
lr_pipeline = PipelineModel.load(MODEL_PATH)
print(f"Loaded LR pipeline with {len(lr_pipeline.stages)} stages.")

Total reviews: 393,890
Loaded LR pipeline with 5 stages.


---
## 3. Building the Comparison Sample

We use the same temporal logic as Notebook 2 (post-2010 = test set) and apply the
same neutral-drop and label rule. Then we sample down to roughly 5,000 reviews
to keep DistilBERT inference under 10 minutes on CPU.

**Why we sample**: DistilBERT is a 67M-parameter transformer. On CPU, even with
batching, it takes a few milliseconds per review. ~500k reviews × a few ms = hours.
On a GPU it's ~10x faster; on a real cluster with multiple GPU workers, the full
dataset becomes feasible. For our project, sampling is the honest trade-off.

In [6]:
# Build the sample — same filtering as Notebook 2's sentiment dataset
sample_size_target = 5_000

sample_df = (
    reviews_df
        .filter(col("Score") != 3)                                      # drop neutrals
        .withColumn("label", F.when(col("Score") >= 4, 1).otherwise(0))
        .withColumn(
            "fullText",
            F.concat_ws(" ", F.coalesce(col("Summary"), F.lit("")), col("Text"))
        )
        .filter(F.year("ReviewDate") >= 2010)                           # same temporal split
        .select("Id", "ProductId", "ReviewDate", "label", "fullText",
                "Score", "HelpfulnessNumerator", "HelpfulnessDenominator",
                "HelpfulnessRatio")
)

# Sample to ~5k reviews. We compute the fraction based on the full test-set size
# so we get roughly the target number regardless of how many rows pass the filter.
total_test_rows = sample_df.count()
fraction = min(1.0, sample_size_target / total_test_rows)
print(f"Test set size after filtering: {total_test_rows:,}")
print(f"Sampling fraction: {fraction:.4f}")

sample_df = sample_df.sample(fraction=fraction, seed=42).cache()
n_sample = sample_df.count()
print(f"Actual sample size: {n_sample:,}")

# Quick check on class balance in the sample
sample_df.groupBy("label").count().orderBy("label").show()

Test set size after filtering: 286,376
Sampling fraction: 0.0175
Actual sample size: 5,106
+-----+-----+
|label|count|
+-----+-----+
|    0|  841|
|    1| 4265|
+-----+-----+



---
## 4. Applying the LR Pipeline

First we get the LR predictions on this sample — straightforward, just call
`.transform()`. This gives us a baseline to compare against.

In [7]:
from pyspark.ml.functions import vector_to_array

# Run the LR pipeline on the sample
lr_predictions = lr_pipeline.transform(sample_df)

# Pull out the columns we'll need later. The probability column is a Spark ML
# vector (VectorUDT), not an array — we need vector_to_array() to index into it.
lr_predictions = (
    lr_predictions
        .withColumnRenamed("prediction", "lr_prediction")
        .withColumn("prob_array", vector_to_array("probability"))
        .withColumn("lr_probability_pos", col("prob_array")[1])
        .select(
            "Id", "ProductId", "ReviewDate", "fullText", "label",
            "Score", "HelpfulnessNumerator", "HelpfulnessDenominator",
            "HelpfulnessRatio",
            "lr_prediction", "lr_probability_pos"
        )
        .cache()
)
lr_predictions.count()
print("LR predictions ready.")
lr_predictions.show(3, truncate=80)

LR predictions ready.
+------+----------+-------------------+--------------------------------------------------------------------------------+-----+-----+--------------------+----------------------+------------------+-------------+------------------+
|    Id| ProductId|         ReviewDate|                                                                        fullText|label|Score|HelpfulnessNumerator|HelpfulnessDenominator|  HelpfulnessRatio|lr_prediction|lr_probability_pos|
+------+----------+-------------------+--------------------------------------------------------------------------------+-----+-----+--------------------+----------------------+------------------+-------------+------------------+
|136515|B006Q820X0|2012-06-08 00:00:00|Good value & good taste I am on auto ship and my 2nd box sb arriving soon.  N...|    1|    5|                   2|                     3|0.6666666666666666|          1.0|0.6696833545851284|
|559987|B004YV7YL4|2012-05-06 00:00:00|Yes, You can Get someth

---
## 5. DistilBERT via `predict_batch_udf`

This is the Lab 9 pattern. The mechanism:

1. We define a **setup function** (`make_sentiment_fn`) that gets called **once per
   worker** at startup. It loads the model and returns an inner `predict` function.
2. `predict_batch_udf` wraps that and gives us back a Spark UDF that processes
   batches of rows.

The win: the model loads once per worker, not once per row. With a 67M-parameter
transformer that matters a lot.

The model is `distilbert-base-uncased-finetuned-sst-2-english` — same one Lab 9
used. It's pretrained on the SST-2 movie sentiment dataset, which is a different
domain from food reviews but close enough that it generalises reasonably well.

In [8]:
from pyspark.ml.functions import predict_batch_udf
from pyspark.sql.types import StructType, StructField, StringType, FloatType


def make_sentiment_fn():
    """
    Called once per worker at startup. Loads the DistilBERT sentiment pipeline
    and returns an inner predict() function that the UDF will call on batches.

    All imports inside the function so the worker (which doesn't share the
    driver's namespace) has everything it needs.
    """
    from transformers import pipeline
    import numpy as np

    # truncation=True + max_length=512: BERT can't handle sequences longer than
    # 512 tokens. Long reviews will be truncated to the first 512 tokens. We could
    # do something cleverer (e.g. average BERT over chunks), but truncation is
    # what Lab 9 did and is fine for a comparison demo.
    classifier = pipeline(
        "sentiment-analysis",
        model="distilbert-base-uncased-finetuned-sst-2-english",
        truncation=True,
        max_length=512,
    )

    def predict(texts: np.ndarray) -> dict:
        # The HuggingFace pipeline expects a list, not a numpy array
        results = classifier(texts.tolist())

        labels      = np.array([r["label"].lower() for r in results])
        confidences = np.array([r["score"]           for r in results], dtype=np.float32)

        # predict_batch_udf wants a dict of arrays when returning a struct
        return {"sentiment_class": labels, "confidence": confidences}

    return predict


# Register the UDF. The return type matches the dict we return above.
sentiment_udf = predict_batch_udf(
    make_sentiment_fn,
    return_type=StructType([
        StructField("sentiment_class", StringType(),  True),    # was False
        StructField("confidence",      FloatType(),   True),    # was False
    ]),
    batch_size=64,
)
print("Sentiment UDF ready.")

Sentiment UDF ready.


### 5.1 Running DistilBERT on the sample

We repartition before applying so each worker gets a chunk to process — same
pattern Lab 9 used for ResNet. `predict_batch_udf` returns a struct, which we
unpack into two plain columns.

In [9]:
# Repartition so that multiple workers process in parallel.
# On Colab (single machine) this gives us multiple processes pulling batches.
bert_predictions = (
    lr_predictions
        .repartition(4)
        .withColumn("sentiment", sentiment_udf(col("fullText")))
        .withColumn("bert_class",      col("sentiment.sentiment_class"))
        .withColumn("bert_confidence", col("sentiment.confidence"))
        .drop("sentiment")
        # Convert BERT's "positive"/"negative" string into a 0/1 label that
        # matches our `label` column
        .withColumn(
            "bert_prediction",
            F.when(col("bert_class") == "positive", 1.0).otherwise(0.0)
        )
        .cache()
)

# Materialise the cache — this is where DistilBERT actually runs
import time
t0 = time.time()
n = bert_predictions.count()
print(f"DistilBERT predictions for {n:,} reviews in {time.time() - t0:.1f}s")

DistilBERT predictions for 5,106 reviews in 52.4s


In [10]:
# Preview both models' predictions side by side
bert_predictions.select(
    "label", "Score", "lr_prediction", "bert_prediction",
    "lr_probability_pos", "bert_confidence", "fullText"
).show(5, truncate=80)

+-----+-----+-------------+---------------+-------------------+---------------+--------------------------------------------------------------------------------+
|label|Score|lr_prediction|bert_prediction| lr_probability_pos|bert_confidence|                                                                        fullText|
+-----+-----+-------------+---------------+-------------------+---------------+--------------------------------------------------------------------------------+
|    1|    5|          1.0|            1.0| 0.9691814233974553|      0.9977962|Wonderful, smooth dark chocolate! There isn't much not to like about this bar...|
|    1|    5|          1.0|            1.0| 0.9997421894697553|     0.99480766|Mmmmm, share the love..... So, I opened the bag just to have a quick taste to...|
|    1|    5|          1.0|            1.0|  0.969631352818036|     0.99799466|delicious A delicious little snack, can also substitute for breakfast in a hu...|
|    1|    4|          1.0|       

---
## 6. Comparing the Two Models

Both models have made predictions on the same reviews. Now we evaluate each
against the ground-truth `label` (derived from Score).

In [11]:
# AUC-ROC for both
# LR uses its probability_pos column; BERT we use confidence × sign
# (positive confidence stays positive, negative gets flipped to 1 - confidence)
bert_for_auc = bert_predictions.withColumn(
    "bert_prob_pos",
    F.when(col("bert_class") == "positive", col("bert_confidence"))
     .otherwise(1.0 - col("bert_confidence"))
)

# We need a rawPrediction-like column for BinaryClassificationEvaluator.
# The evaluator can accept a single probability column via metricName='areaUnderROC'
# if we tell it which column.
auc_evaluator_lr = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="lr_probability_pos",
    metricName="areaUnderROC",
)
auc_evaluator_bert = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="bert_prob_pos",
    metricName="areaUnderROC",
)

auc_lr   = auc_evaluator_lr.evaluate(bert_for_auc)
auc_bert = auc_evaluator_bert.evaluate(bert_for_auc)

print(f"AUC-ROC — LR    : {auc_lr:.4f}")
print(f"AUC-ROC — BERT  : {auc_bert:.4f}")

AUC-ROC — LR    : 0.9445
AUC-ROC — BERT  : 0.9539


In [12]:
# Accuracy and per-class recall — using MulticlassClassificationEvaluator
def model_metrics(pred_col, name):
    """Compute accuracy + per-class recall for a given prediction column."""
    acc_eval = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol=pred_col, metricName="accuracy"
    )
    accuracy = acc_eval.evaluate(bert_predictions)

    recalls = {}
    for cls in [0, 1]:
        rec_eval = MulticlassClassificationEvaluator(
            labelCol="label", predictionCol=pred_col,
            metricName="recallByLabel", metricLabel=float(cls)
        )
        recalls[cls] = rec_eval.evaluate(bert_predictions)

    print(f"{name}:  accuracy={accuracy:.4f}  recall_neg={recalls[0]:.4f}  recall_pos={recalls[1]:.4f}")
    return accuracy, recalls

print("Performance on the test sample:")
acc_lr, rec_lr     = model_metrics("lr_prediction",   "Logistic Regression")
acc_bert, rec_bert = model_metrics("bert_prediction", "DistilBERT")

Performance on the test sample:
Logistic Regression:  accuracy=0.8986  recall_neg=0.8490  recall_pos=0.9083
DistilBERT:  accuracy=0.8670  recall_neg=0.9180  recall_pos=0.8570


**Expected pattern**: LR usually wins or ties on overall accuracy because
the test set is heavily positive-skewed and LR was trained on this exact dataset.
DistilBERT is **zero-shot** here — it never saw a food review during training, only
SST-2 movie reviews. Where DistilBERT often wins is on the minority class (negative
recall) and on subtle cases like negation.

---
## 7. Disagreement Analysis

Where do the two models disagree? This is more interesting than overall metrics
because it tells us *what each model is bringing*.

In [13]:
# Confusion matrix of (LR prediction) vs (BERT prediction)
print("LR prediction × BERT prediction (rows = LR, columns = BERT):")
bert_predictions.groupBy("lr_prediction").pivot("bert_prediction").count().orderBy("lr_prediction").show()

LR prediction × BERT prediction (rows = LR, columns = BERT):
+-------------+---+----+
|lr_prediction|0.0| 1.0|
+-------------+---+----+
|          0.0|856| 249|
|          1.0|526|3475|
+-------------+---+----+



In [14]:
# Disagreement rate
disagree_count = bert_predictions.filter(
    col("lr_prediction") != col("bert_prediction")
).count()
total = bert_predictions.count()

print(f"Total reviews: {total:,}")
print(f"Disagreements: {disagree_count:,} ({100 * disagree_count / total:.2f}%)")

Total reviews: 5,106
Disagreements: 775 (15.18%)


### 7.1 Cases where DistilBERT was right and LR was wrong

These are the ones where the BERT signal actually adds value. We expect this to
include negation-heavy reviews and short, sarcastic ones.

In [15]:
# DistilBERT correct, LR wrong
bert_wins = bert_predictions.filter(
    (col("bert_prediction") == col("label")) &
    (col("lr_prediction")   != col("label"))
).select("label", "Score", "lr_prediction", "bert_prediction",
         "bert_confidence", "fullText")

print(f"Cases where BERT was right and LR was wrong: {bert_wins.count():,}")
bert_wins.show(5, truncate=120)

Cases where BERT was right and LR was wrong: 307
+-----+-----+-------------+---------------+---------------+------------------------------------------------------------------------------------------------------------------------+
|label|Score|lr_prediction|bert_prediction|bert_confidence|                                                                                                                fullText|
+-----+-----+-------------+---------------+---------------+------------------------------------------------------------------------------------------------------------------------+
|    0|    2|          1.0|            0.0|      0.9997092|Spam for vegetarians That's the only thing I could think of when eating this. The texture, the taste, the "gravey" th...|
|    1|    5|          0.0|            1.0|      0.8046912|Toxic Waste It Truly is a Toxic Waste! Really sour , love the flavor. I don't know if this candy is better than warhe...|
|    1|    5|          0.0|            1.0|   

### 7.2 Cases where LR was right and DistilBERT was wrong

The opposite — likely reviews with domain-specific vocabulary that LR learned
from the training data but BERT (trained on movie reviews) doesn't know.

In [16]:
# LR correct, DistilBERT wrong
lr_wins = bert_predictions.filter(
    (col("lr_prediction")   == col("label")) &
    (col("bert_prediction") != col("label"))
).select("label", "Score", "lr_prediction", "bert_prediction",
         "bert_confidence", "fullText")

print(f"Cases where LR was right and BERT was wrong: {lr_wins.count():,}")
lr_wins.show(5, truncate=120)

Cases where LR was right and BERT was wrong: 468
+-----+-----+-------------+---------------+---------------+------------------------------------------------------------------------------------------------------------------------+
|label|Score|lr_prediction|bert_prediction|bert_confidence|                                                                                                                fullText|
+-----+-----+-------------+---------------+---------------+------------------------------------------------------------------------------------------------------------------------+
|    1|    5|          1.0|            0.0|       0.663134|Surprised there are no reviews for this excellently addictive snack I was looking for roasted green peas that were no...|
|    1|    5|          1.0|            0.0|      0.5498408|Far Exceeds the description and photo I was sold on the description and photo that one finds in the listing. But upon...|
|    1|    5|          1.0|            0.0|   

**The takeaway**: each model has a different weakness. LR's strength is its
exposure to our exact training distribution (food-specific vocab); its weakness is
no understanding of context. BERT's strength is contextual reading; its weakness
is no exposure to food-review vocabulary. **An ensemble would probably beat either
alone** — outside the scope of this notebook but worth flagging.

---
## 8. Product-Level Sentiment

Lab 9 used DistilBERT on Shakespeare and aggregated sentiment per play. The same
pattern applies here: aggregate per product. This is useful business output — it
tells the client which products are loved and which need attention.

We compute mean sentiment, a 95% confidence interval, and number of reviews.

In [17]:
# Map BERT's positive/negative to +1 / -1 (Lab 9 pattern)
bert_with_score = bert_predictions.withColumn(
    "bert_score",
    F.when(col("bert_class") == "positive", 1.0).otherwise(-1.0)
)

# Aggregate per product, with confidence intervals
# 95% CI for the mean: mean ± 1.96 * stddev / sqrt(n)
product_sentiment = (
    bert_with_score
        .groupBy("ProductId")
        .agg(
            F.avg("bert_score").alias("avg_sentiment"),
            F.stddev("bert_score").alias("sentiment_stddev"),
            F.count("Id").alias("n_reviews"),
            F.avg("bert_confidence").alias("avg_confidence"),
        )
        .filter(col("n_reviews") >= 10)        # at least 10 reviews to be meaningful
        .withColumn(
            "ci_lower",
            col("avg_sentiment") - 1.96 * col("sentiment_stddev") / F.sqrt(col("n_reviews"))
        )
        .withColumn(
            "ci_upper",
            col("avg_sentiment") + 1.96 * col("sentiment_stddev") / F.sqrt(col("n_reviews"))
        )
)

print("Most positively reviewed products (min 10 reviews in sample):")
product_sentiment.orderBy(F.desc("avg_sentiment")).show(5, truncate=False)

print("Most negatively reviewed products:")
product_sentiment.orderBy("avg_sentiment").show(5, truncate=False)

Most positively reviewed products (min 10 reviews in sample):
+----------+------------------+------------------+---------+------------------+--------------------+------------------+
|ProductId |avg_sentiment     |sentiment_stddev  |n_reviews|avg_confidence    |ci_lower            |ci_upper          |
+----------+------------------+------------------+---------+------------------+--------------------+------------------+
|B003D4F1QS|1.0               |0.0               |11       |0.999100923538208 |1.0                 |1.0               |
|B001EO5Q64|0.8181818181818182|0.6030226891555273|11       |0.9646146514198997|0.4618181818181819  |1.1745454545454546|
|B003B3OOPA|0.6               |0.8432740427115678|10       |0.9982601284980774|0.07733333333333337 |1.1226666666666665|
|B001VJ0B0I|0.6               |0.8432740427115679|10       |0.9874581575393677|0.07733333333333325 |1.1226666666666667|
|B005ZBZLPI|0.4               |0.9660917830792959|10       |0.9719740867614746|-0.1987898908075630

**For the management presentation**: a chart of products with their average
sentiment + error bars is exactly the kind of insight the client cares about.
The confidence interval tells them how reliable each estimate is — a product with
10 reviews has wide bars; one with 200 has tight bars.

---
## 9. BERT Confidence as a Helpfulness Feature (Mini-Experiment)

This is the cross-model dependency I promised in the original plan. The idea:

> *Reviews where BERT is **very confident** about the sentiment might be more useful
> to other customers — they're clearly written, unambiguous. Reviews where BERT is
> uncertain might be confusing or off-topic, and less helpful.*

To test this, we take the subset of our sample with enough helpfulness votes and
train a small helpfulness classifier that includes BERT confidence as a feature.

**Honest caveat**: this is a small-sample demonstration. Our 5k sample, filtered
down to "denominator ≥ 5", will probably leave only 1-2k rows. We can't claim a
definitive improvement at this scale — only show how the BERT signal would be
incorporated. On a real cluster we'd run BERT on the full helpfulness dataset
and the result would be statistically meaningful.

In [18]:
# Subset to reviews with enough helpfulness votes for a reliable label
MIN_VOTES = 5
HELPFUL_THRESHOLD = 0.6

helpful_subset = (
    bert_predictions
        .filter(col("HelpfulnessDenominator") >= MIN_VOTES)
        .withColumn(
            "helpful_label",
            F.when(col("HelpfulnessRatio") > HELPFUL_THRESHOLD, 1).otherwise(0)
        )
        # Build the same engineered features as Notebook 2's helpfulness model
        .withColumn("TextLength", F.length(col("fullText")))
        .withColumn("WordCount",  F.size(F.split(col("fullText"), r"\s+")))
        .withColumn("exclamation_count",
                    F.length(col("fullText")) - F.length(F.regexp_replace(col("fullText"), "!", "")))
        .withColumn("question_count",
                    F.length(col("fullText")) - F.length(F.regexp_replace(col("fullText"), r"\?", "")))
        .withColumn("avg_word_length",
                    F.when(col("WordCount") > 0,
                           col("TextLength") / col("WordCount")).otherwise(0.0))
        .cache()
)

n_subset = helpful_subset.count()
print(f"Reviews in helpfulness subset: {n_subset:,}")
helpful_subset.groupBy("helpful_label").count().orderBy("helpful_label").show()

Reviews in helpfulness subset: 393
+-------------+-----+
|helpful_label|count|
+-------------+-----+
|            0|   94|
|            1|  299|
+-------------+-----+



In [19]:
# Split for the mini-experiment
from pyspark.ml.feature import VectorAssembler, MinMaxScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

train_help, test_help = helpful_subset.randomSplit([0.7, 0.3], seed=42)
print(f"Train: {train_help.count():,}  Test: {test_help.count():,}")

Train: 268  Test: 125


### 9.1 Build two helpfulness models on this small sample

**Model A** — just the structured features (same shape as Notebook 2's model, but
without the Word2Vec embeddings since we don't need them for this comparison).

**Model B** — the same structured features **plus** BERT confidence.

If Model B's AUC is better, the BERT signal is adding value.

In [20]:
STRUCTURED_FEATURES = [
    "Score", "TextLength", "WordCount",
    "exclamation_count", "question_count", "avg_word_length",
]
WITH_BERT_FEATURES = STRUCTURED_FEATURES + ["bert_confidence"]

def build_help_pipeline(feature_cols):
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
    scaler    = MinMaxScaler(inputCol="raw_features", outputCol="features")
    rf        = RandomForestClassifier(
        featuresCol="features",
        labelCol="helpful_label",
        numTrees=100,
        maxDepth=6,
        seed=42,
    )
    return Pipeline(stages=[assembler, scaler, rf])

# Train both
model_a = build_help_pipeline(STRUCTURED_FEATURES).fit(train_help)
model_b = build_help_pipeline(WITH_BERT_FEATURES).fit(train_help)

# Evaluate
help_evaluator = BinaryClassificationEvaluator(
    labelCol="helpful_label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
)

auc_a = help_evaluator.evaluate(model_a.transform(test_help))
auc_b = help_evaluator.evaluate(model_b.transform(test_help))

print(f"Model A (structured only)          : AUC = {auc_a:.4f}")
print(f"Model B (structured + BERT signal) : AUC = {auc_b:.4f}")
print(f"Delta: {auc_b - auc_a:+.4f}")

Model A (structured only)          : AUC = 0.7644
Model B (structured + BERT signal) : AUC = 0.8147
Delta: +0.0503


In [21]:
# Feature importances from Model B — does BERT confidence rank highly?
rf_b = model_b.stages[-1]
importances = rf_b.featureImportances.toArray()

print("Feature importances (Model B):")
for name, imp in sorted(zip(WITH_BERT_FEATURES, importances), key=lambda x: x[1], reverse=True):
    bar = "█" * int(imp * 200)
    print(f"  {name:<22}  {imp:.4f}  {bar}")

Feature importances (Model B):
  Score                   0.3186  ███████████████████████████████████████████████████████████████
  bert_confidence         0.1511  ██████████████████████████████
  avg_word_length         0.1349  ██████████████████████████
  TextLength              0.1314  ██████████████████████████
  WordCount               0.1260  █████████████████████████
  exclamation_count       0.0867  █████████████████
  question_count          0.0514  ██████████


**Interpretation**: on this small sample the result is suggestive rather than
definitive. Even a small improvement in AUC indicates BERT confidence is picking
up signal beyond what the structured features capture. With more data we'd
expect this gap to be more reliable and likely larger.

The bigger picture: combining a domain-trained model with a pretrained transformer
gives the helpfulness model access to **two different views** of each review — its
surface characteristics (length, punctuation) and its semantic clarity (BERT
confidence). That's the kind of feature engineering you can only do once you've
built both models.

---
## 10. Wrap-Up

### What we built
1. Applied a pretrained **DistilBERT** sentiment model at scale on Spark using
   `predict_batch_udf`.
2. Compared it head-to-head against our Notebook 2 Logistic Regression classifier
   on the same test sample.
3. Looked at disagreements to see where each model wins.
4. Aggregated sentiment per product (Lab 9's per-play pattern adapted).
5. Mini-experiment showing how BERT confidence can serve as an extra feature for
   the helpfulness model.

### Key Spark functionalities demonstrated
- **`predict_batch_udf`** — the right way to run a heavy model at scale on Spark.
  Loads the model once per worker, processes batches.
- **Pipeline model reloading** — `PipelineModel.load()` lets us hand off a trained
  model between notebooks (and would let us hand it off between training and
  serving in production).
- **Cross-model feature engineering** — using one model's output as input to
  another.

### Honest limitations
- **We sampled** — ~5,000 reviews instead of the full ~150k post-2010. CPU
  inference time was the constraint. With a GPU runtime in Colab (or proper
  cluster GPU workers) we'd run the full data and get statistically tighter
  comparisons.
- **DistilBERT is out-of-domain** — pretrained on SST-2 (movie reviews), applied
  to food reviews. Fine-tuning it on Amazon reviews would push accuracy higher
  but is a bigger project on its own.


1. Use `predict_batch_udf` to extract DistilBERT embeddings (not the SST-2 head).
2. Save the embeddings to Parquet.
3. Train a small classification head on those embeddings using
   `TorchDistributor.run(...)` — exactly as Lab 9 did with the flower-head model.

We left this as an extension — the existing DistilBERT inference already meets
the brief's deep-learning requirement.

### Next Notebook
- **Notebook 4**: GraphFrames analysis of the user-product review network.


In [22]:
# Cleanup
for df in [sample_df, lr_predictions, bert_predictions, helpful_subset]:
    try:
        df.unpersist()
    except Exception:
        pass

spark.stop()
print("Spark session stopped.")

Spark session stopped.
